In [5]:
import json
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def output_results(results_list):
  
  # print(results_list)
  data_list = []
  for result in results_list:
    y_test = result['y_test']
    y_pred = result['y_pred']
    config = result['configuration_details']

    file_name = config["classifier"] + "_data.json"
    
    accuracy = accuracy_score(y_test, y_pred)
    report_dict = classification_report(
        y_test,
        y_pred,
        target_names=['heart_disease_no', 'heart_disease_yes'],
        output_dict=True)
    cf_matrix = confusion_matrix(y_test, y_pred)
    data = {
        "configuration_details": config,
        "classification_report": report_dict,
        "Accuracy": str(accuracy),
        "tn": str(cf_matrix[0][0]),
        "fp": str(cf_matrix[0][1]),
        "fn": str(cf_matrix[1][0]),
        "tp": str(cf_matrix[1][1]),

    }
    data_list.append(data)
  with open(file_name, "w") as file:
      json.dump(data_list, file, indent=4)

## Random Forest Classifier

In [27]:
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

X_train_final, y_train_final = joblib.load('X_train_y_train.pkl')
X_test_final, y_test = joblib.load('X_test_y_test.pkl')

result_list = []
clf = RandomForestClassifier(random_state=42)

In [28]:
# grid search to optimize parameters
grid_params = {
    'n_estimators': [25, 50, 75, 100, 125, 150],
    'max_depth': list(range(2, 10)), 
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

grid_search = GridSearchCV(clf, grid_params, cv=5, scoring='recall')
grid_search.fit(X_train_final, y_train_final)

best_grid_model = grid_search.best_estimator_

y_pred_grid = best_grid_model.predict(X_test_final)

result_list.append({
    "y_test": y_test,
    "y_pred": y_pred_grid,
    "configuration_details": {
        "classifier": "RandomForestClassifier",
        "search_type": "GridSearch",
        "best_params": grid_search.best_params_,
    }
})


In [29]:
# randomized search to optimize parameters
random_params = {
    'n_estimators': [25, 50, 75, 100, 125, 150],
    'max_depth': list(range(2, 10)), 
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(clf, random_params, n_iter=10, cv=5, scoring='recall', random_state=42)
random_search.fit(X_train_final, y_train_final)

best_random_model = random_search.best_estimator_

y_pred_random = best_random_model.predict(X_test_final)

result_list.append({
    "y_test": y_test,
    "y_pred": y_pred_random,
    "configuration_details": {
        "classifier": "RandomForestClassifier",
        "search_type": "RandomSearch",
        "best_params": random_search.best_params_,
    }
})

output_results(result_list)

In [30]:
import joblib

# Save best GridSearch model
joblib.dump(best_grid_model, "best_random_forest_grid_model.pkl")

# Save best RandomSearch model
joblib.dump(best_random_model, "best_random_forest_random_model.pkl")


['best_random_forest_random_model.pkl']

## Naive Bayes Classifier

In [21]:
from sklearn.naive_bayes import GaussianNB
import joblib
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

X_train_final, y_train_final = joblib.load('X_train_y_train.pkl')
X_test_final, y_test = joblib.load('X_test_y_test.pkl')

result_list = []
model = GaussianNB()


In [ ]:

grid_params = {
    'var_smoothing': [1e-16, 1e-15, 1e-14, 1e-13, 1e-12, 1e-11, 1e-10, 1e-9]
}

grid_search = GridSearchCV(model, grid_params, cv=5, scoring='recall')
grid_search.fit(X_train_final, y_train_final)

best_grid_model = grid_search.best_estimator_

y_pred_grid = best_grid_model.predict(X_test_final)

result_list.append({
    "y_test": y_test,
    "y_pred": y_pred_grid,
    "configuration_details": {
      "classifier": "GaussianNB",
      "best_params": grid_search.best_params_,
    }
})


In [ ]:
# randomized search to optimize parameters
random_params = {
    'var_smoothing': [1e-16, 1e-15, 1e-14, 1e-13, 1e-12, 1e-11, 1e-10, 1e-9]
}


random_search = RandomizedSearchCV(model, random_params, n_iter=10, cv=5, scoring='recall', random_state=42)
random_search.fit(X_train_final, y_train_final)

best_random_model = random_search.best_estimator_

y_pred_random = best_random_model.predict(X_test_final)

result_list.append({
    "y_test": y_test,
    "y_pred": y_pred_random,
    "configuration_details": {
      "classifier": "GaussianNB",
      "best_params": random_search.best_params_,
    }
})

output_results(result_list)

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 4 is smaller than n_iter=10. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


## Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

result_list = []
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train_final, y_train_final)
y_pred = model.predict(X_test_final)
result_list.append({
    "y_test": y_test,
    "y_pred": y_pred
})
configuration = {
  "classifier": "DecisionTreeClassifier",

}
output_results(result_list)

## KNN Classifier

In [106]:
from sklearn.neighbors import KNeighborsClassifier

result_list = []
model = KNeighborsClassifier()
model.fit(X_train_final, y_train_final)
y_pred = model.predict(X_test_final)
result_list.append({
    "y_test": y_test,
    "y_pred": y_pred
})
configuration = {
  "classifier": "knn",
  "neighbors": str(5),
  "weights": "uniform",
  "smote": {
    "applied": "yes",
    "strategy": str(smpl_strategy),
    "nearest_neighbors": str(k_nabes)
  }
}
output_results(result_list, configuration)

## SVC

In [107]:
from sklearn.svm import SVC

result_list = []
model = SVC(random_state=42)
model.fit(X_train_final, y_train_final)
y_pred = model.predict(X_test_final)
result_list.append({
    "y_test": y_test,
    "y_pred": y_pred
})
configuration = {
  "classifier": "SVC",
  "smote": {
    "applied": "yes",
    "strategy": str(smpl_strategy),
    "nearest_neighbors": str(k_nabes)
  }
}
output_results(result_list, configuration)

## Logistic Regression

In [108]:
from sklearn.linear_model import LogisticRegression

result_list = []
model = LogisticRegression(random_state=42)
model.fit(X_train_final, y_train_final)
y_pred = model.predict(X_test_final)
result_list.append({
    "y_test": y_test,
    "y_pred": y_pred
})
configuration = {
  "classifier": "LogisticRegression",
  "smote": {
    "applied": "yes",
    "strategy": str(smpl_strategy),
    "nearest_neighbors": str(k_nabes)
  }
}
output_results(result_list, configuration)

## MLP Classifier

In [111]:
from sklearn.neural_network import MLPClassifier

result_list = []
model = MLPClassifier(random_state=42, max_iter=300)
model.fit(X_train_final, y_train_final)
y_pred = model.predict(X_test_final)
result_list.append({
    "y_test": y_test,
    "y_pred": y_pred
})
configuration = {
  "classifier": "MLPClassifier",
  "max_iter": str(300),
  "smote": {
    "applied": "yes",
    "strategy": str(smpl_strategy),
    "nearest_neighbors": str(k_nabes)
  }
}
output_results(result_list, configuration)

/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
